# AtliQ Hardware — Sales Insights Analysis

**Dataset:** AtliQ Hardware sales data (Codebasics)  
**Database:** MySQL `sales` — cleaned via the `sales_cleaned` view  
**Goal:** Understand revenue trends, top performers, and channel dynamics to surface actionable business insights.

---

## Table of Contents
1. [Setup & Data Load](#1-setup--data-load)
2. [Dataset Overview](#2-dataset-overview)
3. [Revenue Trend Analysis](#3-revenue-trend-analysis)
4. [Year-over-Year Growth](#4-year-over-year-growth)
5. [Top Customers](#5-top-customers)
6. [Top Products](#6-top-products)
7. [Market & Regional Performance](#7-market--regional-performance)
8. [Channel Mix (Brick & Mortar vs E-Commerce)](#8-channel-mix)
9. [Rolling Averages & Smoothed Trend](#9-rolling-averages--smoothed-trend)
10. [Customer Concentration (Pareto Analysis)](#10-customer-concentration-pareto-analysis)
11. [Key Findings Summary](#11-key-findings-summary)

## 1. Setup & Data Load

We use the shared `db.py` helper (which reads credentials from `.env`) to pull the `sales_cleaned` view into Pandas.  
All revenue figures are in **INR** — USD transactions were converted at a fixed rate of ₹83/$ in the view.

In [ ]:
import sys
import os

# Allow imports from the project root regardless of where Jupyter is launched
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dashboard.db import query

# Pandas display settings
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)

In [ ]:
# Load the entire cleaned view into memory (~150k rows after cleaning)
df = query("SELECT * FROM sales_cleaned")

# Ensure correct dtypes
df['order_date'] = pd.to_datetime(df['order_date'])
df['year']       = df['year'].astype(int)

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
df.head()

## 2. Dataset Overview

Before any analysis, validate the cleaned data looks as expected.

In [ ]:
print("=== Shape ===")
print(f"{len(df):,} transactions | {df['year'].nunique()} years | "
      f"{df['customer_code'].nunique()} customers | "
      f"{df['markets_name'].nunique()} markets | "
      f"{df['product_code'].nunique()} products")

print("\n=== Date Range ===")
print(f"{df['order_date'].min().date()}  →  {df['order_date'].max().date()}")

print("\n=== Revenue (INR) ===")
print(f"Total: ₹{df['sales_amount_inr'].sum():,.0f}")
print(f"Mean per transaction: ₹{df['sales_amount_inr'].mean():,.2f}")

print("\n=== Currency split ===")
print(df['currency'].value_counts())

print("\n=== Nulls ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Revenue Trend Analysis

We look at revenue at two grains: **annual** (to spot multi-year direction) and **monthly** (to identify seasonality).

In [ ]:
# --- Annual revenue ---
annual = (
    df.groupby('year', as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'),
           transactions=('sales_amount_inr', 'count'),
           units_sold=('sales_qty', 'sum'))
      .sort_values('year')
)
print(annual.to_string(index=False))

In [ ]:
fig = px.bar(
    annual, x='year', y='revenue_inr',
    text=annual['revenue_inr'].apply(lambda v: f'₹{v/1e7:.1f}Cr'),
    title='Annual Revenue (INR)',
    labels={'revenue_inr': 'Revenue (INR)', 'year': 'Year'},
    color='revenue_inr',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, showlegend=False)
fig.show()

In [ ]:
# --- Monthly revenue ---
MONTH_ORDER = [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
]

monthly = (
    df.groupby(['year', 'month_name'], as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'))
)
monthly['month_num'] = monthly['month_name'].apply(
    lambda m: MONTH_ORDER.index(m) + 1
)
monthly = monthly.sort_values(['year', 'month_num'])
monthly['period'] = monthly['year'].astype(str) + '-' + monthly['month_num'].astype(str).str.zfill(2)

fig = px.line(
    monthly, x='period', y='revenue_inr',
    color='year', markers=True,
    title='Monthly Revenue by Year',
    labels={'revenue_inr': 'Revenue (INR)', 'period': 'Month'}
)
fig.update_xaxes(tickangle=45)
fig.show()

## 4. Year-over-Year Growth

Using `pct_change()` on annual revenue — equivalent to the `LAG()` window function in `04_kpi_queries.sql`.

In [ ]:
annual['yoy_growth_pct'] = annual['revenue_inr'].pct_change() * 100
annual['yoy_growth_pct'] = annual['yoy_growth_pct'].round(2)
print(annual[['year', 'revenue_inr', 'yoy_growth_pct']].to_string(index=False))

In [ ]:
yoy = annual.dropna(subset=['yoy_growth_pct'])

fig = px.bar(
    yoy, x='year', y='yoy_growth_pct',
    text=yoy['yoy_growth_pct'].apply(lambda v: f'{v:+.1f}%'),
    title='Year-over-Year Revenue Growth (%)',
    labels={'yoy_growth_pct': 'YoY Growth (%)', 'year': 'Year'},
    color='yoy_growth_pct',
    color_continuous_scale='RdYlGn'
)
fig.update_traces(textposition='outside')
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 5. Top Customers

Which customers drive the most revenue, and what is their share of the total?

In [ ]:
top_customers = (
    df.groupby(['customer_code', 'customer_name', 'customer_type'], as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'),
           transactions=('sales_amount_inr', 'count'))
      .sort_values('revenue_inr', ascending=False)
)
top_customers['revenue_share_pct'] = (
    top_customers['revenue_inr'] / top_customers['revenue_inr'].sum() * 100
).round(2)

print(top_customers.head(10).to_string(index=False))

In [ ]:
top5_cust = top_customers.head(5)

fig = px.bar(
    top5_cust, x='customer_name', y='revenue_inr',
    color='customer_type',
    text=top5_cust['revenue_share_pct'].apply(lambda v: f'{v:.1f}%'),
    title='Top 5 Customers by Revenue',
    labels={'revenue_inr': 'Revenue (INR)', 'customer_name': 'Customer'}
)
fig.update_traces(textposition='outside')
fig.show()

## 6. Top Products

Which products generate the most revenue? Are any product types dominant?

In [ ]:
top_products = (
    df.groupby(['product_code', 'product_type'], as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'),
           units_sold=('sales_qty', 'sum'))
      .sort_values('revenue_inr', ascending=False)
)
top_products['revenue_share_pct'] = (
    top_products['revenue_inr'] / top_products['revenue_inr'].sum() * 100
).round(2)

top5_prod = top_products.head(5)
print(top5_prod.to_string(index=False))

In [ ]:
fig = px.bar(
    top5_prod, x='product_code', y='revenue_inr',
    color='product_type',
    text=top5_prod['revenue_share_pct'].apply(lambda v: f'{v:.1f}%'),
    title='Top 5 Products by Revenue',
    labels={'revenue_inr': 'Revenue (INR)', 'product_code': 'Product'}
)
fig.update_traces(textposition='outside')
fig.show()

## 7. Market & Regional Performance

AtliQ operates across North, South, and Central India, plus two international locations (New York, Paris).

In [ ]:
markets = (
    df.groupby(['markets_name', 'zone'], as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'),
           transactions=('sales_amount_inr', 'count'))
      .sort_values('revenue_inr', ascending=False)
)
markets['revenue_share_pct'] = (
    markets['revenue_inr'] / markets['revenue_inr'].sum() * 100
).round(2)

print(markets.to_string(index=False))

In [ ]:
fig = px.bar(
    markets, x='markets_name', y='revenue_inr',
    color='zone',
    text=markets['revenue_share_pct'].apply(lambda v: f'{v:.1f}%'),
    title='Revenue by Market',
    labels={'revenue_inr': 'Revenue (INR)', 'markets_name': 'Market'},
    category_orders={'markets_name': markets['markets_name'].tolist()}
)
fig.update_traces(textposition='outside')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Zone-level summary
zones = (
    df.groupby('zone', as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'))
      .sort_values('revenue_inr', ascending=False)
)

fig = px.pie(
    zones, names='zone', values='revenue_inr',
    title='Revenue Share by Zone',
    hole=0.4
)
fig.show()

## 8. Channel Mix

How does the **Brick & Mortar** vs **E-Commerce** split evolve year over year?
A shift toward E-Commerce would indicate a change in AtliQ's go-to-market strategy.

In [ ]:
channel = (
    df.groupby(['year', 'customer_type'], as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'))
)
channel_total = channel.groupby('year')['revenue_inr'].transform('sum')
channel['channel_share_pct'] = (channel['revenue_inr'] / channel_total * 100).round(2)

print(channel.sort_values(['year', 'customer_type']).to_string(index=False))

In [ ]:
fig = px.bar(
    channel, x='year', y='channel_share_pct',
    color='customer_type', barmode='group',
    text=channel['channel_share_pct'].apply(lambda v: f'{v:.1f}%'),
    title='Channel Mix by Year (% of Annual Revenue)',
    labels={'channel_share_pct': 'Share (%)', 'year': 'Year', 'customer_type': 'Channel'}
)
fig.update_traces(textposition='outside')
fig.show()

## 9. Rolling Averages & Smoothed Trend

A 3-month rolling average removes short-term noise and makes directional trends easier to read.
This metric would appear on an executive dashboard as a leading indicator.

In [ ]:
# Build a proper time-indexed monthly series
monthly_ts = (
    df.groupby(df['order_date'].dt.to_period('M'))
      .agg(revenue_inr=('sales_amount_inr', 'sum'))
      .reset_index()
)
monthly_ts['order_date'] = monthly_ts['order_date'].dt.to_timestamp()
monthly_ts = monthly_ts.sort_values('order_date')
monthly_ts['rolling_3m'] = monthly_ts['revenue_inr'].rolling(window=3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_ts['order_date'], y=monthly_ts['revenue_inr'],
    name='Monthly Revenue', marker_color='lightblue', opacity=0.7
))
fig.add_trace(go.Scatter(
    x=monthly_ts['order_date'], y=monthly_ts['rolling_3m'],
    name='3-Month Rolling Avg', line=dict(color='darkblue', width=2)
))
fig.update_layout(
    title='Monthly Revenue with 3-Month Rolling Average',
    xaxis_title='Month', yaxis_title='Revenue (INR)'
)
fig.show()

## 10. Customer Concentration (Pareto Analysis)

The Pareto principle (80/20 rule) asks: what fraction of customers generates 80% of revenue?
High concentration is a business risk — losing one top customer causes an outsized revenue drop.

In [ ]:
pareto = (
    df.groupby('customer_name', as_index=False)
      .agg(revenue_inr=('sales_amount_inr', 'sum'))
      .sort_values('revenue_inr', ascending=False)
      .reset_index(drop=True)
)
total_rev = pareto['revenue_inr'].sum()
pareto['cumulative_pct'] = (pareto['revenue_inr'].cumsum() / total_rev * 100).round(2)
pareto['customer_rank']  = range(1, len(pareto) + 1)
pareto['customer_pct']   = (pareto['customer_rank'] / len(pareto) * 100).round(2)

# How many customers make up 80% of revenue?
cutoff = pareto[pareto['cumulative_pct'] <= 80]
print(f"{len(cutoff)} out of {len(pareto)} customers ({len(cutoff)/len(pareto)*100:.1f}%) "
      f"generate {cutoff['revenue_inr'].sum()/total_rev*100:.1f}% of revenue")

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(x=pareto['customer_name'], y=pareto['revenue_inr'],
           name='Revenue (INR)', marker_color='steelblue'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=pareto['customer_name'], y=pareto['cumulative_pct'],
               name='Cumulative %', line=dict(color='tomato', width=2)),
    secondary_y=True
)
fig.add_hline(y=80, line_dash='dash', line_color='grey',
              annotation_text='80% threshold', secondary_y=True)

fig.update_layout(title='Customer Revenue Pareto Chart',
                  xaxis_tickangle=45)
fig.update_yaxes(title_text='Revenue (INR)', secondary_y=False)
fig.update_yaxes(title_text='Cumulative %', secondary_y=True)
fig.show()

## 11. Key Findings Summary

> **Fill this section in after running all cells against your live database.**  
> The template below lists the questions each section answered — replace the placeholders with your actual numbers.

| # | Finding | Value |
|---|---------|-------|
| 1 | Total revenue (all years, INR) | ₹ ___ Cr |
| 2 | Peak revenue year | 20__ |
| 3 | Worst YoY decline | __% (20__ → 20__) |
| 4 | Top customer by revenue | ___ (___ % share) |
| 5 | Top product by revenue | ___ (___ % share) |
| 6 | Highest-revenue market | ___ |
| 7 | Dominant zone | ___ (___ % share) |
| 8 | E-Commerce share trend | Growing / Declining / Stable |
| 9 | Customers generating 80% of revenue | ___ out of ___ |
| 10 | Biggest data quality issue found | ___ |

### Recommended Actions
- **Customer concentration risk:** If top-N customers account for >50% of revenue, prioritise diversification.
- **Declining markets:** Review markets with negative YoY trends for pricing or distribution issues.
- **Channel mix:** If E-Commerce share is growing, consider reallocating marketing spend.
- **Seasonal peaks:** Use the monthly chart to plan inventory and promotions.